In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

data_raw = pd.read_csv('1.data/housing.csv')

# долгота, широта, жилищный_средний_возраст, всего комнат, общее количество спален, население
# домохозяйства, медианный_доход, медианная_стоимость_дома, 
# близость океана разбиваем на колонки one hot encoding и выкидываем одну из них впоследствии (для пользы модели, чтобы колонки в сумме давали разное влияние)
# <1H OCEAN # INLAND # NEAR OCEAN # NEAR BAY # ISLAND

data_encoded = pd.get_dummies(data_raw, columns=['ocean_proximity'], drop_first=True) #убираем одну колонку чтобы убрать влияние мультиколонности на модель

# Убираем пропуски в данных в спальнях
median = data_encoded['total_bedrooms'].median() # Считаем медиану по всей колонке
data_encoded['total_bedrooms'] = data_encoded['total_bedrooms'].fillna(median) # Меняем пропущенные значения на медиану 

# Отделяем признаки (X) от целевой переменной (y)
X = data_encoded.drop('median_house_value', axis=1)  # Все колонки, кроме цены
y = data_encoded['median_house_value']               # Только цена

# Разбиваем: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,          # 30% в тест
    random_state=42         # Фиксируем "случайность", чтобы результат был воспроизводим
)
print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

# Создаем масштабатор
scaler_x = StandardScaler()
scaler_y = StandardScaler()

# Обучаем масштабатор на тренировочных данных и преобразуем их
# Для каждого значения x в признаке применяется формула Z‑оценки: z=(x−μ)/σ​
# где: μ — среднее значение признака (mean), σ — стандартное отклонение признака (std).
# После преобразования: новое среднее ≈0, новая дисперсия =1.

# Стандартное отклонение считается кв. корень из дисперсии D(X) =sum(xi-average)²/ n мера степени изменчивости значений переменной относительно её среднего

X_train_scaled = scaler_x.fit_transform(X_train)
Y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)) # reshape(-1, 1) переводим массив в двумерный тензор

# Преобразуем ТЕСТОВЫЕ данные (используем тот же масштаб)
X_test_scaled = scaler_x.transform(X_test)
Y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

print(f"Среднее X_train после масштабирования: {X_train_scaled.mean(axis=0)[:5]}")
print(f"Стандартное отклонение X_train: {X_train_scaled.std(axis=0)[:5]}")
print(f"Среднее y_train после масштабирования: {Y_train_scaled.mean():.6f}")
print(f"Стандартное отклонение y_train: {Y_train_scaled.std():.6f}")

# Создаем тензоры
x_train_tensor = torch.tensor(X_train_scaled.astype('float32'), dtype=torch.float32)
x_test_tensor = torch.tensor(X_test_scaled.astype('float32'), dtype=torch.float32)

y_train_tensor = torch.tensor(Y_train_scaled.astype('float32'), dtype=torch.float32)
y_test_tensor = torch.tensor(Y_test_scaled.astype('float32'), dtype=torch.float32)

BATCH_SIZE = 32

# Создание датасета
dataset = TensorDataset(x_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True)

input_features = X_train.shape[1]
print(f"Количество признаков: {input_features}")

# Модель (линейная регрессия)
model = torch.nn.Sequential(torch.nn.Linear(in_features=input_features, out_features=1)) # 12 признаков на входе, 1 цена на выходе

# Обучение. Функция потерь и оптимизатор
loss = torch.nn.MSELoss(reduction='mean') # Средняя квадратичная ошибка (Mean Squared Error)
trainer = torch.optim.SGD(model.parameters(), lr=0.001)

for epoch in range(200):
    for batch_X, batch_y in dataloader:
        pred = model(batch_X)           # Прямой проход (forward)
        l = loss(pred, batch_y)         # Функция потерь. Одно число (среднее по батчу)
        
        trainer.zero_grad() # Обнуляем градиенты (иначе они накапливаются)
        l.backward()        # Обратный проход. Считаем градиент усреднен по батчу
        trainer.step()      # Обновляем веса
    
    if epoch % 10 == 0:
        print(f"Эпоха {epoch}, Потеря: {l.item():.4f}")

# Тест
print("Оценка модели на тестовых данных:")
with torch.no_grad():
    test_pred_scaled = model(x_test_tensor)
    
    # Возвращаем масштаб, чтобы получить цены в долларах
    test_pred = scaler_y.inverse_transform(test_pred_scaled.numpy())
    y_test_original = scaler_y.inverse_transform(y_test_tensor.numpy())
    
    # Считаем ошибку в долларах
    mse_dollars = mean_squared_error(y_test_original, test_pred)
    rmse_dollars = np.sqrt(mse_dollars)
    
    print(f"MSE (в долларах²): {mse_dollars:.2f}")
    print(f"RMSE (в долларах): {rmse_dollars:.2f}")


Размер обучающей выборки: 14448
Размер тестовой выборки: 6192
Среднее X_train после масштабирования: [-3.24190041e-15 -1.46382230e-15 -5.55726254e-17  1.42620012e-17
 -1.19013941e-16]
Стандартное отклонение X_train: [1. 1. 1. 1. 1.]
Среднее y_train после масштабирования: 0.000000
Стандартное отклонение y_train: 1.000000
Количество признаков: 12
Эпоха 0, Потеря: 0.3292
Эпоха 10, Потеря: 0.4779
Эпоха 20, Потеря: 0.8392
Эпоха 30, Потеря: 0.2846
Эпоха 40, Потеря: 0.6714
Эпоха 50, Потеря: 0.4353
Эпоха 60, Потеря: 0.4044
Эпоха 70, Потеря: 0.5397
Эпоха 80, Потеря: 0.1731
Эпоха 90, Потеря: 0.5011
Эпоха 100, Потеря: 0.1960
Эпоха 110, Потеря: 0.4253
Эпоха 120, Потеря: 0.5563
Эпоха 130, Потеря: 0.2639
Эпоха 140, Потеря: 0.4945
Эпоха 150, Потеря: 0.7435
Эпоха 160, Потеря: 0.2570
Эпоха 170, Потеря: 0.1862
Эпоха 180, Потеря: 0.3461
Эпоха 190, Потеря: 0.2587
Оценка модели на тестовых данных:
MSE (в долларах²): 4726222848.00
RMSE (в долларах): 68747.53
